# Poshtetgarm — Google Colab Job Runner

نوت‌بوک عمومی برای اجرای هر job از پوشه `tools/colab/jobs/` روی سرور Google Colab (بدون اشغال فضای لپ‌تاپ).

**گردش کار:**
1. **سلول ۱:** بررسی GPU/CPU و مونت اختیاری Google Drive (برای کش شدن مدل‌ها بین سشن‌ها).
2. **سلول ۲:** آپلود اسکریپت `jobs/<job>.py` و فایل ورودی (مثل `starter-tts-input.json`).
3. **سلول ۳:** اجرای job با آرگومان‌های دلخواه.
4. **سلول ۴:** دانلود فایل خروجی (`output.zip`) و ذخیره اختیاری در Google Drive.

In [ ]:
#@title 1) Runtime check & optional Google Drive cache
MOUNT_DRIVE = True  #@param {type:"boolean"}

import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path('/content/work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() if gpu.returncode == 0 else 'None (CPU runtime)')
total, used, free = shutil.disk_usage('/content')
print(f'Disk free: {free / 1e9:.1f} GB')

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/poshtetgarm-colab')
    CACHE_DIR = DRIVE_ROOT / 'cache'
    OUTPUTS_DIR = DRIVE_ROOT / 'outputs'
else:
    CACHE_DIR = Path('/content/colab_cache')
    OUTPUTS_DIR = WORK_DIR / 'outputs'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_DIR:   ', WORK_DIR)
print('CACHE_DIR:  ', CACHE_DIR)
print('OUTPUTS_DIR:', OUTPUTS_DIR)

In [ ]:
#@title 2) Upload job script (e.g. kokoro_tts.py) and input file (e.g. starter-tts-input.json)
from google.colab import files as colab_files

uploaded = colab_files.upload()
for name, data in uploaded.items():
    dest = WORK_DIR / name
    dest.write_bytes(data)
    print(f'Saved {dest} ({len(data) // 1024} KB)')

In [ ]:
#@title 3) Run job
JOB_SCRIPT = "kokoro_tts.py"  #@param {type:"string"}
INPUT_FILE = "starter-tts-input.json"  #@param {type:"string"}
OUTPUT_NAME = "poshtetgarm_output.zip"  #@param {type:"string"}
EXTRA_ARGS = "--voice af_bella --speed 0.95"  #@param {type:"string"}

import shlex
import subprocess
import sys

script_path = WORK_DIR / JOB_SCRIPT
output_path = WORK_DIR / OUTPUT_NAME
cmd = [sys.executable, str(script_path), '--cache-dir', str(CACHE_DIR), '--output', str(output_path)]
if INPUT_FILE.strip():
    cmd.extend(['--input', str(WORK_DIR / INPUT_FILE.strip())])
if EXTRA_ARGS.strip():
    cmd.extend(shlex.split(EXTRA_ARGS))

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
#@title 4) Save copy to Drive (if mounted) & download output
import shutil
from google.colab import files as colab_files

assert output_path.exists(), f'Output not found: {output_path}'
if MOUNT_DRIVE:
    drive_copy = OUTPUTS_DIR / output_path.name
    shutil.copy2(output_path, drive_copy)
    print('Copied to Drive:', drive_copy)

colab_files.download(str(output_path))